# Tarea: Análisis de "El ramo azul" de Octavio Paz
**Curso:** Analítica de Textos y Modelos de Lenguaje      
**Alumno:** Luis David Aguilar Colorado    
**Herramientas utilizadas:** NLTK, spaCy y Stanza     
**Fecha:** 15/02/2026

In [1]:
import nltk
import spacy
import stanza

# Descargas necesarias para NLTK
nltk.download('punkt_tab') # Para tokenizar palabras y oraciones del archivo .txt
nltk.download('book')      # Visto en clase para los corpus de ejemplo

# Modelo en español para Stanza
stanza.download('es')

c:\Users\hazar\anaconda3\envs\analisis_texto\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hazar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading collection 'book'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\hazar\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package brown to
[nltk_data]    |     C:\Users\hazar\AppData\Roaming\nltk_data...
[nltk_data]    |   Package brown is already up-to-date!
[nltk_data]    | Downloading package chat80 to
[nltk_data]    |     C:\Users\hazar\AppData\Roaming\nltk_data...
[nltk_data]    |   Package chat80 is already up-to-date!
[nltk_data]  

> **Nota de spaCy:** A diferencia de NLTK y Stanza, el modelo en español de spaCy (`es_core_news_sm`) no se descarga con código de Python, debe ser instalado previamente desde la terminal ejecutando: `python -m spacy download es_core_news_sm`

In [2]:
# 1. Leer el archivo local
f = open('elramoazul.txt', 'r', encoding='utf-8')
texto_raw = f.read()
f.close()

# 2. Inicializar herramientas

# spaCy: Cargar el modelo en español
nlp_spacy = spacy.load('es_core_news_sm') 
doc_spacy = nlp_spacy(texto_raw)

# Stanza: Cargar el pipeline en español
nlp_stanza = stanza.Pipeline(lang='es', processors='tokenize,mwt,pos,lemma')
doc_stanza = nlp_stanza(texto_raw)

2026-02-22 17:32:50 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-02-22 17:32:50 INFO: Downloaded file to C:\Users\hazar\stanza_resources\resources.json
2026-02-22 17:32:50 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-02-22 17:32:50 INFO: Using device: cpu
2026-02-22 17:32:50 INFO: Loading: tokenize
2026-02-22 17:32:54 INFO: Loading: mwt
2026-02-22 17:32:54 INFO: Loading: pos
2026-02-22 17:32:57 INFO: Loading: lemma
2026-02-22 17:32:58 INFO: Done loading processors!


## Pregunta 1 - ¿Cuantas palabras hay en el texto? (descartando símbolos de puntuación)

Para contar las palabras descartando los signos de puntuación, se usara el método `word_tokenize` de NLTK. 

In [3]:
# Conteo con NLTK
tokens_nltk = nltk.word_tokenize(texto_raw)
# Usar .isalpha() para descartar signos de puntuación y números
palabras_nltk = [w for w in tokens_nltk if w.isalpha()]
total_nltk = len(palabras_nltk)
print(f"Total según NLTK:   {total_nltk}")

# Conteo con spaCy
# Basandose en los atributos del token (descartando puntuación y espacios)
palabras_spacy = [token.text for token in doc_spacy if not token.is_punct and not token.is_space]
total_spacy = len(palabras_spacy)
print(f"Total según spaCy:  {total_spacy}")

# Conteo con Stanza
# Basandose en el Universal POS tag (se descarta 'PUNCT')
palabras_stanza = [word.text for sent in doc_stanza.sentences for word in sent.words if word.upos != 'PUNCT']
total_stanza = len(palabras_stanza)
print(f"Total según Stanza: {total_stanza}")

Total según NLTK:   800
Total según spaCy:  826
Total según Stanza: 849


Aunque las tres herramientas son útiles, **spaCy** ofrece la respuesta más cercana al conteo ortográfico tradicional (826 palabras) debido a que su tokenizador está optimizado para español.   

El conteo inferior de **NLTK** (800) se debe a que está diseñado principalmente para inglés y su tokenizador puede dejar signos de puntuación adheridos a las palabras.

El conteo superior de **Stanza** (849) ocurre porque esta herramienta aplica la Expansión de Tokens Multi-Palabra (MWT), separando contracciones gramaticales (como "al" en "a" + "el" o "del" en "de" + "el"), incrementando el número total.

## Pregunta 2 - ¿Cuantas palabras diferentes hay?

Para encontrar las palabras diferentes (vocabulario), se convierte todo a minúsculas y despues se usa función `set()` para obtener los elementos únicos.

In [4]:
# NLTK
vocabulario_nltk = set([w.lower() for w in palabras_nltk])
total_dif_nltk = len(vocabulario_nltk)
print(f"Vocabulario según NLTK:   {total_dif_nltk}")

# spaCy
vocabulario_spacy = set([w.lower() for w in palabras_spacy])
total_dif_spacy = len(vocabulario_spacy)
print(f"Vocabulario según spaCy:  {total_dif_spacy}")

# Stanza
vocabulario_stanza = set([w.lower() for w in palabras_stanza])
total_dif_stanza = len(vocabulario_stanza)
print(f"Vocabulario según Stanza: {total_dif_stanza}")

Vocabulario según NLTK:   422
Vocabulario según spaCy:  442
Vocabulario según Stanza: 434


Al igual que en el conteo total, el tamaño del vocabulario varía por cómo cada herramienta tokeniza el texto. 

* **NLTK** muestra el vocabulario más reducido porque omitió palabras enteras durante el filtrado de puntuación con `.isalpha()`.
* **spaCy** agrupa las palabras basándose en su forma ortográfica estándar, dando el conteo más tradicional.
* **Stanza** puede mostrar variaciones en los elementos únicos debido a su separación de tokens multi-palabra (MWT)

## Pregunta 3: Después de lematizar las palabras, ¿cuántas palabras diferentes hay?

La lematización reduce las palabras a su forma base de diccionario (lema), ya que `WordNetLemmatizer` de NLTK se utiliza para inglés, se usaran spaCy y Stanza para extraer los lemas correctos en español.

In [5]:
# spaCy
# Extraer el lema si el token no es puntuación
lemas_spacy = [token.lemma_.lower() for token in doc_spacy if not token.is_punct and token.is_alpha]
lemas_unicos_spacy = len(set(lemas_spacy))
print(f"Palabras diferentes tras lematizar (spaCy): {lemas_unicos_spacy}")

# Stanza
# Iteracion por oraciones y luego por palabras
lemas_stanza = [word.lemma.lower() for sent in doc_stanza.sentences for word in sent.words if word.upos != 'PUNCT']
lemas_unicos_stanza = len(set(lemas_stanza))
print(f"Palabras diferentes tras lematizar (Stanza): {lemas_unicos_stanza}")

Palabras diferentes tras lematizar (spaCy): 377
Palabras diferentes tras lematizar (Stanza): 373


## Pregunta 4: ¿Cuál es la diversidad léxica del texto dado? (relación de palabras únicas con respecto al número total de palabras)

La  diversidad léxica de un texto se calcula dividiendo el total de palabras unicas entre el numero total de palabras.

In [6]:
# Diversidad Léxica
# Palabras únicas (Pregunta 2) / Total de palabras (Pregunta 1)
diversidad_nltk = total_dif_nltk / total_nltk
print(f"Diversidad según NLTK:   {diversidad_nltk:.4f}")

diversidad_spacy = total_dif_spacy / total_spacy
print(f"Diversidad según spaCy:  {diversidad_spacy:.4f}")

diversidad_stanza = total_dif_stanza / total_stanza
print(f"Diversidad según Stanza: {diversidad_stanza:.4f}")

Diversidad según NLTK:   0.5275
Diversidad según spaCy:  0.5351
Diversidad según Stanza: 0.5112


## Pregunta 5: ¿Cuáles son las 20 palabras (únicas) más frecuentes en el texto? ¿Cuál es su frecuencia?

In [7]:
from collections import Counter

# Se agrupa el nombre y la lista de palabras de cada herramienta
herramientas = [
    ("NLTK", palabras_nltk),
    ("spaCy", palabras_spacy),
    ("Stanza", palabras_stanza)
]

for nombre, palabras in herramientas:
    # Convertir a minúsculas
    palabras_min = [w.lower() for w in palabras]
    
    # Contar las frecuencias (FreqDist para NLTK y Counter para las demás)
    contador = nltk.FreqDist(palabras_min) if nombre == "NLTK" else Counter(palabras_min)
    
    # Imprimir de forma limpia
    top_20_texto = [f"{palabra}: {frec}" for palabra, frec in contador.most_common(20)]
    
    print(f"\nPara {nombre}:")
    print(" | ".join(top_20_texto))


Para NLTK:
de: 37 | la: 36 | me: 24 | el: 22 | y: 20 | un: 16 | no: 16 | los: 16 | a: 13 | ojos: 13 | una: 11 | con: 11 | se: 10 | que: 10 | al: 9 | en: 7 | las: 7 | mis: 7 | del: 6 | señor: 6

Para spaCy:
de: 37 | la: 36 | me: 24 | el: 22 | y: 20 | un: 16 | no: 16 | los: 16 | a: 13 | ojos: 13 | una: 11 | con: 11 | se: 10 | que: 10 | al: 9 | en: 7 | las: 7 | mis: 7 | del: 6 | señor: 6

Para Stanza:
de: 42 | el: 36 | la: 36 | me: 32 | a: 22 | y: 20 | no: 20 | un: 16 | los: 16 | ojos: 13 | una: 11 | se: 11 | con: 11 | que: 10 | mis: 8 | en: 7 | las: 7 | señor: 6 | entre: 5 | lo: 5


## Pregunta 6: ¿Cuál es el número promedio de palabras por oración?

Para esto, se necesita dividir el total de palabras entre el número de oraciones.

In [8]:
# Oraciones y promedio con NLTK
oraciones_nltk = nltk.sent_tokenize(texto_raw)
num_oraciones_nltk = len(oraciones_nltk)
promedio_nltk = total_nltk / num_oraciones_nltk 
print(f"NLTK:   {num_oraciones_nltk} oraciones | Promedio: {promedio_nltk:.2f} palabras/oración")


# Oraciones y promedio con spaCy
# spaCy genera un iterador, debe convertirce a lista para contar
oraciones_spacy = list(doc_spacy.sents)
num_oraciones_spacy = len(oraciones_spacy)
promedio_spacy = total_spacy / num_oraciones_spacy
print(f"spaCy:  {num_oraciones_spacy} oraciones | Promedio: {promedio_spacy:.2f} palabras/oración")


# Oraciones y promedio con Stanza
oraciones_stanza = doc_stanza.sentences
num_oraciones_stanza = len(oraciones_stanza)
promedio_stanza = total_stanza / num_oraciones_stanza
print(f"Stanza: {num_oraciones_stanza} oraciones | Promedio: {promedio_stanza:.2f} palabras/oración")

NLTK:   105 oraciones | Promedio: 7.62 palabras/oración
spaCy:  101 oraciones | Promedio: 8.18 palabras/oración
Stanza: 106 oraciones | Promedio: 8.01 palabras/oración


## Pregunta 7: ¿Cuál es la frecuencia de sustantivos, adjetivos y verbos en el texto?

Para el etiquetado de Partes de la Oración (POS Tagging), se descarta el uso de NLTK, su etiquetador nativo no soporta español y se centra en el inglés.    
En su lugar, se prefiere usar **spaCy** y **Stanza**, implementan el conjunto de etiquetas **UNIVERSAL**, permitiendo extraer directamente las frecuencias buscando las etiquetas `NOUN` (Sustantivos), `ADJ` (Adjetivos) y `VERB` (Verbos).

In [9]:
from collections import Counter

# POS Tagging con spaCy
# Extraer y contar todas las etiquetas (pos_)
conteo_spacy = Counter(token.pos_ for token in doc_spacy)
print("\nspaCy:")
print(f"- Sustantivos (NOUN): {conteo_spacy.get('NOUN', 0)}")
print(f"- Adjetivos (ADJ):    {conteo_spacy.get('ADJ', 0)}")
print(f"- Verbos (VERB):      {conteo_spacy.get('VERB', 0)}")


# POS Tagging con Stanza
# Extraer y contar todas las etiquetas (upos)
conteo_stanza = Counter(word.upos for sent in doc_stanza.sentences for word in sent.words)
print("\nStanza:")
print(f"- Sustantivos (NOUN): {conteo_stanza.get('NOUN', 0)}")
print(f"- Adjetivos (ADJ):    {conteo_stanza.get('ADJ', 0)}")
print(f"- Verbos (VERB):      {conteo_stanza.get('VERB', 0)}")


spaCy:
- Sustantivos (NOUN): 180
- Adjetivos (ADJ):    66
- Verbos (VERB):      132

Stanza:
- Sustantivos (NOUN): 183
- Adjetivos (ADJ):    60
- Verbos (VERB):      148
